# Telco Troubleshooting Agentic — Benchmark Notebook

Run the full evaluation pipeline (**Track A**, **Track B**, or **both**) on Colab or Kaggle.

---

### Quick start
1. Run **Cell 1** (platform setup + repo clone)
2. Run **Cell 2** (install dependencies)
3. **Edit `BenchmarkConfig` in Cell 3** to switch between `dev` / `prod` and tune all run parameters
4. Run the remaining cells in order

### Environment modes

| Mode | LLM backend | When to use |
|------|-------------|-------------|
| `dev`  | Ollama (local, `qwen3.5:35b-a3b`) | Free, local experimentation |
| `prod` | OpenRouter (cloud API) | Competition submission, benchmarking with a strong model |

## Cell 1 — Platform Setup & Repository Clone

In [1]:
"""
Platform detection — works on Colab, Kaggle, and local Jupyter.
Clones the repo (using a GitHub PAT on cloud) and adds it to sys.path.
"""
import os
import sys
from pathlib import Path

# ── Detect platform ──────────────────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    PLATFORM = "colab"
except ImportError:
    PLATFORM = "kaggle" if os.path.exists("/kaggle/working") else "local"

print(f"Platform: {PLATFORM}")

# ── GitHub PAT ────────────────────────────────────────────────────────────────
# On Colab: add a secret named GITHUB_PAT via the 🔑 Secrets panel.
# On Kaggle: add it under Add-ons → Secrets → GITHUB_PAT.
# Locally:  set the GITHUB_PAT env var or leave empty for public repo access.
GITHUB_PAT: str = ""

if PLATFORM == "colab":
    try:
        from google.colab import userdata
        GITHUB_PAT = userdata.get("GITHUB_PAT") or ""
    except Exception:
        pass
elif PLATFORM == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        GITHUB_PAT = UserSecretsClient().get_secret("GITHUB_PAT") or ""
    except Exception:
        pass
else:
    GITHUB_PAT = os.getenv("GITHUB_PAT", "")

# ── Paths ─────────────────────────────────────────────────────────────────────
if PLATFORM == "colab":
    REPO_DIR = Path("/content/Troubleshooting-Agentic")
elif PLATFORM == "kaggle":
    REPO_DIR = Path("/kaggle/working/Troubleshooting-Agentic")
else:
    # Running locally inside the repo — go up one level from notebooks/
    REPO_DIR = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

REPO_DIR = REPO_DIR.resolve()

# ── Build authenticated clone URL ─────────────────────────────────────────────
_REPO_SLUG = "Seydifa/Troubleshooting-Agentic"

if GITHUB_PAT:
    REPO_URL = f"https://{GITHUB_PAT}@github.com/{_REPO_SLUG}.git"
    print("GitHub PAT found — using authenticated clone URL.")
else:
    REPO_URL = f"https://github.com/{_REPO_SLUG}.git"
    print("No GitHub PAT — using public clone URL (may fail for private repos).")

# ── Clone repo (cloud only) ────────────────────────────────────────────────────
if PLATFORM in ("colab", "kaggle"):
    if not REPO_DIR.exists():
        print(f"Cloning {_REPO_SLUG} → {REPO_DIR} …")
        ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
        if ret != 0:
            raise RuntimeError("git clone failed. Check your GITHUB_PAT and repo name.")
    else:
        print(f"Repo already present at {REPO_DIR}, pulling latest …")
        # Inject credentials into the existing remote before pulling
        _auth_remote = REPO_URL if GITHUB_PAT else f"https://github.com/{_REPO_SLUG}.git"
        os.system(f"git -C {REPO_DIR} remote set-url origin {_auth_remote}")
        os.system(f"git -C {REPO_DIR} pull --ff-only")
else:
    print(f"Running locally — using repo at {REPO_DIR}")

# ── Add repo root to Python path ───────────────────────────────────────────────
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")
print("Done.")

Platform: colab
GitHub PAT found — using authenticated clone URL.
Cloning Seydifa/Troubleshooting-Agentic → /content/Troubleshooting-Agentic …
Working directory: /content/Troubleshooting-Agentic
Done.


## Cell 2 — Install Dependencies

In [2]:
import subprocess, sys

def pip_install(*packages: str) -> None:
    """Install packages quietly, skipping those already present."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        stdout=subprocess.DEVNULL,
    )

print("Installing project dependencies …")
pip_install("-r", "requirements.txt")

# Extra cloud convenience packages
pip_install("fastapi", "uvicorn[standard]", "flask")

print("All dependencies installed.")

Installing project dependencies …
All dependencies installed.


## Cell 3 — Benchmark Configuration (edit here)

> **This is the only cell you need to edit.**  
> Set `ENV` to `"dev"` (Ollama, free) or `"prod"` (OpenRouter, cloud API).  
> All other values have sensible defaults — override only what you need.

In [19]:
from dataclasses import dataclass, field
from typing import Literal, Optional
from google.colab import userdata
import os


@dataclass
class BenchmarkConfig:
    """
    Central configuration object for the benchmark run.

    Edit the fields below, then run the next cell to apply them.
    Call `cfg.use_dev()` or `cfg.use_prod()` for a one-liner switch.
    """

    # ── Environment ────────────────────────────────────────────────────────────
    # "dev"  → Ollama (free, local/Colab)
    # "prod" → OpenRouter (cloud API, requires openrouter_api_key)
    ENV: Literal["dev", "prod"] = "dev"

    # ── Track selection ────────────────────────────────────────────────────────
    # "A"    → 5G RF drive-test troubleshooting
    # "B"    → IP network topology / fault diagnosis
    # "both" → run both tracks
    TRACK: Literal["A", "B", "both"] = "both"

    # ── Dev (Ollama) settings ──────────────────────────────────────────────────
    OLLAMA_BASE_URL: str = "http://localhost:11434"
    MODEL_NAME: str = "qwen3.5:35b-a3b"
    PARSER_MODEL_NAME: str = "qwen3.5:35b-a3b"

    # ── Prod (OpenRouter) settings ────────────────────────────────────────────
    # Get a key at https://openrouter.ai — add it here or as a Colab/Kaggle secret
    OPENROUTER_API_KEY: str = ""
    OPENROUTER_BASE_URL: str = "https://openrouter.ai/api/v1"
    PROD_MODEL_NAME: str = "Qwen/Qwen3.5-35B-A3B"

    # ── Tool server ────────────────────────────────────────────────────────────
    TOOL_SERVER_URL: str = "http://localhost:8000"

    # ── HuggingFace ────────────────────────────────────────────────────────────
    # Required to download the gated dataset.
    # Add your token at https://huggingface.co/settings/tokens
    HF_TOKEN: str = userdata.get("HF_TOKEN") or ""

    # ── Run parameters ─────────────────────────────────────────────────────────
    PHASE: Literal["1", "2"] = "1"
    # Set to an integer to process only the first N scenarios (dev smoke-test).
    # Set to None to process all scenarios (full benchmark).
    LIMIT: Optional[int] = 10
    VERBOSE: bool = False

    # ── Paths ──────────────────────────────────────────────────────────────────
    TEST_FILE_A: str = "data/raw/Track A/data/Phase_1/test.json"
    TEST_FILE_B: str = "data/raw/Track B/data/Phase_1/test.json"
    TRAIN_FILE_A: str = "data/raw/Track A/data/Phase_1/train.json"
    RAG_INDEX: str = "rag_index.pkl"
    TOOL_CACHE: str = "tool_cache_a.pkl"
    OUTPUT_CSV: str = "result.csv"

    # ── Optional build flags ───────────────────────────────────────────────────
    # True → rebuild RAG index from train.json before running (slow, once-only)
    BUILD_RAG: bool = False
    # True → pre-warm the Track A tool server cache before inference
    WARM_CACHE: bool = False

    # ── Convenience setters ────────────────────────────────────────────────────

    def use_dev(
        self,
        model: str = "qwen3.5:35b-a3b",
        ollama_url: str = "http://localhost:11434",
        limit: Optional[int] = 5,
    ) -> "BenchmarkConfig":
        """Switch to dev (Ollama) mode. Returns self for chaining."""
        self.ENV = "dev"
        self.MODEL_NAME = model
        self.PARSER_MODEL_NAME = model
        self.OLLAMA_BASE_URL = ollama_url
        self.LIMIT = limit
        print(f"[BenchmarkConfig] dev mode — model={model}, limit={limit}")
        return self

    def use_prod(
        self,
        api_key: str = "",
        model: str = "Qwen/Qwen3.5-35B-A3B",
        limit: Optional[int] = None,
    ) -> "BenchmarkConfig":
        """Switch to prod (OpenRouter) mode. Returns self for chaining."""
        self.ENV = "prod"
        self.OPENROUTER_API_KEY = api_key or self.OPENROUTER_API_KEY
        self.PROD_MODEL_NAME = model
        self.LIMIT = limit
        print(f"[BenchmarkConfig] prod mode — model={model}, limit={limit or 'ALL'}")
        return self

    def full_run(self) -> "BenchmarkConfig":
        """Remove the scenario limit to run the full benchmark."""
        self.LIMIT = None
        print("[BenchmarkConfig] Full run — limit removed")
        return self

    def smoke_test(self, n: int = 3) -> "BenchmarkConfig":
        """Restrict to the first N scenarios for a quick smoke-test."""
        self.LIMIT = n
        print(f"[BenchmarkConfig] Smoke-test — limit={n}")
        return self

    def summary(self) -> None:
        """Print a human-readable summary of the current configuration."""
        print("=" * 55)
        print("  BenchmarkConfig summary")
        print("=" * 55)
        print(f"  ENV          : {self.ENV}")
        print(f"  TRACK        : {self.TRACK}")
        if self.ENV == "dev":
            print(f"  Model        : {self.MODEL_NAME} (Ollama @ {self.OLLAMA_BASE_URL})")
        else:
            print(f"  Model        : {self.PROD_MODEL_NAME} (OpenRouter)")
            key_preview = (self.OPENROUTER_API_KEY[:6] + "…") if self.OPENROUTER_API_KEY else "NOT SET"
            print(f"  API key      : {key_preview}")
        print(f"  Tool server  : {self.TOOL_SERVER_URL}")
        print(f"  Phase        : {self.PHASE}")
        print(f"  Limit        : {self.LIMIT or 'ALL scenarios'}")
        print(f"  Verbose      : {self.VERBOSE}")
        print(f"  BUILD_RAG    : {self.BUILD_RAG}")
        print(f"  WARM_CACHE   : {self.WARM_CACHE}")
        print(f"  Output CSV   : {self.OUTPUT_CSV}")
        print("=" * 55)


# ── Instantiate and customise ──────────────────────────────────────────────────
cfg = BenchmarkConfig()

# ---- EDIT BELOW THIS LINE -------------------------------------------------- #

# Option A: quick one-liner switch
# cfg.use_dev()                        # free, local/Colab Ollama
# cfg.use_prod(api_key="sk-or-...")    # OpenRouter cloud

# Option B: fine-grained control
cfg.ENV          = "dev"              # "dev" | "prod"
cfg.TRACK        = "both"            # "A" | "B" | "both"
cfg.LIMIT        = 6                 # None for full run
cfg.VERBOSE      = True
cfg.HF_TOKEN     = ""                # fill in if running --download
cfg.OPENROUTER_API_KEY = ""          # fill in for prod mode

# ---- END EDIT ZONE --------------------------------------------------------- #

cfg.summary()

  BenchmarkConfig summary
  ENV          : dev
  TRACK        : both
  Model        : qwen3.5:35b-a3b (Ollama @ http://localhost:11434)
  Tool server  : http://localhost:8000
  Phase        : 1
  Limit        : 6
  Verbose      : True
  BUILD_RAG    : False
  WARM_CACHE   : False
  Output CSV   : result.csv


## Cell 4 — Apply Configuration

Pushes all `BenchmarkConfig` values to environment variables and patches the `settings` singleton so every module picks up the correct values.

In [20]:
"""
Applies the BenchmarkConfig to:
1. os.environ — so dotenv-less code and subprocesses see the right values.
2. src.config.settings — the live singleton used by llm.py / orchestrator.py.
3. src.llm  — clears the LLM singletons so they are re-created with new settings.
"""

import importlib

# ── 1. Write to environment ───────────────────────────────────────────────────
_model_for_env = cfg.PROD_MODEL_NAME if cfg.ENV == "prod" else cfg.MODEL_NAME

_env_map = {
    "ENV":                  cfg.ENV,
    "TRACK":                cfg.TRACK,
    "MODEL_NAME":           _model_for_env,
    "PARSER_MODEL_NAME":    cfg.PARSER_MODEL_NAME if cfg.ENV == "dev" else _model_for_env,
    "OLLAMA_BASE_URL":      cfg.OLLAMA_BASE_URL,
    "OPENROUTER_API_KEY":   cfg.OPENROUTER_API_KEY,
    "OPENROUTER_BASE_URL":  cfg.OPENROUTER_BASE_URL,
    "TOOL_SERVER_URL":      cfg.TOOL_SERVER_URL,
    "HF_TOKEN":             cfg.HF_TOKEN,
    "TEST_FILE_A_PATH":     cfg.TEST_FILE_A,
    "TEST_FILE_B_PATH":     cfg.TEST_FILE_B,
    "OUTPUT_CSV_PATH":      cfg.OUTPUT_CSV,
    "RAG_INDEX_PATH":       cfg.RAG_INDEX,
    "TOOL_CACHE_PATH":      cfg.TOOL_CACHE,
}

for key, value in _env_map.items():
    if value:  # don't overwrite with empty strings
        os.environ[key] = str(value)

# ── 2. Patch the live settings singleton ─────────────────────────────────────
# Re-import config so the singleton reflects the updated env vars.
import src.config as _cfg_module
importlib.reload(_cfg_module)

# Patch the module-level singleton that all other src.* modules imported
from src.config import Settings as _Settings
_new_settings = _Settings()
_cfg_module.settings = _new_settings

# Re-export into the global namespace for convenience
settings = _new_settings

# ── 3. Clear LLM singletons ───────────────────────────────────────────────────
# Forces get_reasoning_llm() / get_parser_llm() to re-create instances that
# point to the correct Ollama URL / OpenRouter key on the next call.
try:
    import src.llm as _llm_module
    _llm_module.clear_llm_cache()
except Exception:
    pass  # src.llm not yet importable (first run before deps installed)

print(f"Settings applied — ENV={settings.env}, model={settings.model_name}")


Settings applied — ENV=dev, model=qwen3.5:35b-a3b


## Cell 4.5 — Weights & Biases Setup (optional)

Log every scenario result in real time to a W&B table so you can watch the
agent's output **while the run is still going** — useful for long 400+ scenario
runs where you want to catch issues early without waiting for the full result.

**Setup**:
1. `pip install wandb` (already in `requirements.txt`)
2. Add a Colab secret named `WANDB_API_KEY` **or** set `cfg.WANDB_API_KEY` below
3. If `WANDB_API_KEY` is empty, logging is silently disabled — nothing breaks

The table columns logged per scenario:
`idx · scenario_id · track · answer · ground_truth · exact_match · problem_type · retry_count · status · elapsed_s · reasoning_preview`


In [ ]:
from src.wandb_logger import WandbLogger

# ── W&B API key: Colab secret → cfg field → env var ──────────────────────────
_wandb_key = ""
if PLATFORM == "colab":
    try:
        from google.colab import userdata
        _wandb_key = userdata.get("WANDB_API_KEY") or ""
    except Exception:
        pass
if not _wandb_key:
    _wandb_key = os.environ.get("WANDB_API_KEY", "")
if _wandb_key:
    os.environ["WANDB_API_KEY"] = _wandb_key

# ── Initialise logger ─────────────────────────────────────────────────────────
# Set enabled=False to disable all W&B logging without changing anything else.
wl = WandbLogger(
    project="telco-troubleshooting",
    run_name=f"benchmark-{cfg.TRACK}-{cfg.ENV}",
    config={
        "track":      cfg.TRACK,
        "env":        cfg.ENV,
        "model":      cfg.MODEL_NAME if cfg.ENV == "dev" else cfg.PROD_MODEL_NAME,
        "limit":      cfg.LIMIT,
        "phase":      cfg.PHASE,
    },
    flush_every=10,          # log table snapshot every 10 scenarios
    enabled=bool(_wandb_key),
)
print("W&B logger ready." if _wandb_key else "W&B disabled (no WANDB_API_KEY).")


## Cell 5 — (dev only) Ollama Setup

Skip this cell entirely when `cfg.ENV == "prod"`.

On **Colab / Kaggle** in dev mode, this installs Ollama, starts the daemon, and pulls the configured model. On a local machine with Ollama already running, it just verifies connectivity.

In [21]:
# ── Install Ollama (Colab / Kaggle only) ──────────────────────────────────────
# Run this cell once per session. Safe to re-run — skips install if already present.

import os, shutil
import time

if os.path.exists("/kaggle/working") or "google.colab" in str(globals()):
    if shutil.which("ollama") is None:
        # ── Ollama: install + serve + pull models ──────────────────────────
        os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
        os.system('apt-get install -y -q zstd')
        os.system('curl -fsSL https://ollama.com/install.sh | sh')

        ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
        if not os.path.isfile(ollama_bin):
            raise RuntimeError('Ollama install failed — binary not found.')

        subprocess.Popen([ollama_bin, 'serve'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)
        print(f"Ollama installed: {shutil.which('ollama')}")
    else:
        print(f"Ollama already installed: {shutil.which('ollama')}")
else:
    print("Local environment — skipping Ollama install (assumed already available).")

Ollama already installed: /usr/local/bin/ollama


In [22]:
import shutil
import subprocess
import time
import requests

if cfg.ENV != "dev":
    print("Skipping Ollama setup (prod mode — using OpenRouter).")
else:
    ollama_url = cfg.OLLAMA_BASE_URL

    # Known install locations used by the official Ollama install script
    _OLLAMA_CANDIDATE_PATHS = [
        "/usr/local/bin/ollama",
        "/usr/bin/ollama",
        os.path.expanduser("~/.local/bin/ollama"),
    ]

    def _resolve_ollama_bin() -> str | None:
        """Return the path to the ollama binary, or None if not found."""
        found = shutil.which("ollama")
        if found:
            return found
        for p in _OLLAMA_CANDIDATE_PATHS:
            if os.path.isfile(p) and os.access(p, os.X_OK):
                return p
        return None

    def _ollama_alive(url: str, timeout: float = 3.0) -> bool:
        try:
            return requests.get(f"{url}/api/tags", timeout=timeout).status_code == 200
        except Exception:
            return False

    if not _ollama_alive(ollama_url):
        if PLATFORM in ("colab", "kaggle"):
            print("Ollama not running — installing …")
            ret = os.system("curl -fsSL https://ollama.ai/install.sh | sh")
            if ret != 0:
                raise RuntimeError("Ollama install script failed.")

            # Refresh PATH so shutil.which can find the new binary
            _new_paths = ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]
            for _p in _new_paths:
                if _p not in os.environ.get("PATH", ""):
                    os.environ["PATH"] = _p + ":" + os.environ.get("PATH", "")

            ollama_bin = _resolve_ollama_bin()
            if ollama_bin is None:
                raise RuntimeError(
                    "Ollama binary not found after installation. "
                    f"Checked: PATH + {_OLLAMA_CANDIDATE_PATHS}"
                )
            print(f"Ollama binary found at: {ollama_bin}")

            # Start Ollama as a background daemon using the resolved path
            subprocess.Popen(
                [ollama_bin, "serve"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )

            # Wait until the API is up (up to 30 s)
            print("Waiting for Ollama daemon to start …", end="", flush=True)
            for _ in range(30):
                if _ollama_alive(ollama_url):
                    print(" ready.")
                    break
                print(".", end="", flush=True)
                time.sleep(1)
            else:
                raise RuntimeError("Ollama daemon did not start within 30 s.")
            print("Ollama daemon started.")
        else:
            raise RuntimeError(
                f"Ollama is not reachable at {ollama_url}.\n"
                "Start it with: `ollama serve`"
            )
    else:
        print(f"Ollama is already running at {ollama_url}.")

    # ── Pull the model if not present ─────────────────────────────────────────
    model = cfg.MODEL_NAME
    print(f"Checking model '{model}' …")
    tags = requests.get(f"{ollama_url}/api/tags").json().get("models", [])
    present = any(m.get("name", "").startswith(model.split(":")[0]) for m in tags)
    if not present:
        print(f"Pulling {model} (this may take a while on first run) …")
        resp = requests.post(
            f"{ollama_url}/api/pull",
            json={"name": model},
            stream=True,
            timeout=600,
        )
        for line in resp.iter_lines():
            if line:
                import json as _json
                msg = _json.loads(line)
                status = msg.get("status", "")
                if "pulling" in status or "success" in status:
                    print(f"  {status}")
    else:
        print(f"Model '{model}' already present.")

    print("Ollama ready.")

Ollama is already running at http://localhost:11434.
Checking model 'qwen3.5:35b-a3b' …
Model 'qwen3.5:35b-a3b' already present.
Ollama ready.


## Cell 6 — Dataset Download

Downloads the gated HuggingFace dataset. **Requires `cfg.HF_TOKEN`** and that you have accepted the dataset terms at [huggingface.co/datasets/netop/Telco-Troubleshooting-Agentic-Challenge](https://huggingface.co/datasets/netop/Telco-Troubleshooting-Agentic-Challenge).

Skip this cell if `data/raw/` is already populated.

In [23]:
from pathlib import Path

_test_a = Path(cfg.TEST_FILE_A)
_test_b = Path(cfg.TEST_FILE_B)
_both_present = _test_a.exists() and _test_b.exists()

if _both_present:
    print("Dataset already present — skipping download.")
    print(f"  Track A test: {_test_a}")
    print(f"  Track B test: {_test_b}")
else:
    # ── Resolve HF_TOKEN: cfg field → platform secrets → env var ─────────────
    _hf_token = cfg.HF_TOKEN

    if not _hf_token and PLATFORM == "colab":
        try:
            from google.colab import userdata
            _hf_token = userdata.get("HF_TOKEN") or ""
        except Exception:
            pass

    if not _hf_token and PLATFORM == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            _hf_token = UserSecretsClient().get_secret("HF_TOKEN") or ""
        except Exception:
            pass

    if not _hf_token:
        _hf_token = os.environ.get("HF_TOKEN", "")

    if not _hf_token:
        raise ValueError(
            "HF_TOKEN is not set. Either:\n"
            "  • Set cfg.HF_TOKEN = '...' in Cell 3, or\n"
            "  • Add a GITHUB secret named HF_TOKEN in Colab (🔑) / Kaggle (Add-ons → Secrets)"
        )

    # ── Push token into env + live settings singleton ─────────────────────────
    os.environ["HF_TOKEN"] = _hf_token
    cfg.HF_TOKEN = _hf_token
    settings.hf_token = _hf_token  # patch the singleton used by src/download.py

    print("Downloading dataset from HuggingFace …")
    from src.download import download as _hf_download
    _hf_download()
    print("Download complete.")

Dataset already present — skipping download.
  Track A test: data/raw/Track A/data/Phase_1/test.json
  Track B test: data/raw/Track B/data/Phase_1/test.json


## Cell 7 — Start Tool Servers

Track A uses a **FastAPI** server and Track B uses a **Flask** server, both on port 8000.
The cell starts whichever server(s) are required, waits for them to become healthy, and confirms readiness.

In [24]:
import subprocess
import time
import requests
from pathlib import Path

_server_proc: subprocess.Popen | None = None


def _server_healthy(url: str, timeout: float = 3.0) -> bool:
    try:
        return requests.get(url, timeout=timeout).status_code < 500
    except Exception:
        return False


def _start_server(track: str, server_url: str) -> subprocess.Popen | None:
    """Start the appropriate tool server for the given track."""
    if _server_healthy(server_url):
        print(f"Tool server already healthy at {server_url}")
        return None

    if track == "A":
        server_dir = Path("data/raw/Track A")
        cmd = [
            sys.executable, "-m", "uvicorn", "server:app",
            "--host", "0.0.0.0", "--port", "8000",
        ]
    else:  # B
        server_dir = Path("data/raw/Track B")
        cmd = [sys.executable, "server.py"]

    if not server_dir.exists():
        raise FileNotFoundError(
            f"Server directory not found: {server_dir}\n"
            "Run Cell 6 (dataset download) first."
        )

    print(f"Starting Track {track} tool server (cwd={server_dir}) …")
    proc = subprocess.Popen(
        cmd,
        cwd=server_dir,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # Wait up to 20 s for the server to become healthy
    for _ in range(20):
        if _server_healthy(server_url):
            print(f"Track {track} server is up at {server_url}")
            return proc
        time.sleep(1)

    proc.terminate()
    raise RuntimeError(f"Track {track} server did not become healthy within 20 s.")


# Start whichever server(s) the selected track requires.
# Both tracks share the same port, so only one can run at a time when TRACK != "both".
# For "both", Track A starts first; Track B reuses the same port after Track A finishes.
# (In practice the competition evaluation runs them sequentially.)

_server_url = cfg.TOOL_SERVER_URL

if cfg.TRACK in ("A", "both"):
    _server_proc = _start_server("A", _server_url)
elif cfg.TRACK == "B":
    _server_proc = _start_server("B", _server_url)

print("Tool server(s) ready.")

Starting Track A tool server (cwd=data/raw/Track A) …
Track A server is up at http://localhost:8000
Tool server(s) ready.


## Cell 8 — Load Test Data

In [25]:
import json

def _load_json(path: str) -> list:
    with open(path, encoding="utf-8") as fh:
        data = json.load(fh)
    if isinstance(data, dict):
        return data.get("data") or data.get("scenarios") or data.get("questions") or []
    return data


test_data: list = []

if cfg.TRACK in ("A", "both"):
    _scenarios_a = _load_json(cfg.TEST_FILE_A)
    for s in _scenarios_a:
        s["track"] = "A"
    test_data.extend(_scenarios_a)
    print(f"Track A: {len(_scenarios_a)} scenarios loaded from {cfg.TEST_FILE_A}")

if cfg.TRACK in ("B", "both"):
    _scenarios_b = _load_json(cfg.TEST_FILE_B)
    for s in _scenarios_b:
        s["track"] = "B"
    test_data.extend(_scenarios_b)
    print(f"Track B: {len(_scenarios_b)} scenarios loaded from {cfg.TEST_FILE_B}")

# Apply limit
if cfg.LIMIT is not None:
    test_data = test_data[: cfg.LIMIT]
    print(f"Limiting to first {cfg.LIMIT} scenarios (smoke-test mode)")

print(f"\nTotal scenarios to process: {len(test_data)}")

Track A: 500 scenarios loaded from data/raw/Track A/data/Phase_1/test.json
Track B: 50 scenarios loaded from data/raw/Track B/data/Phase_1/test.json
Limiting to first 6 scenarios (smoke-test mode)

Total scenarios to process: 6


## Cell 9 — (Optional) Build RAG Index

Only required when `cfg.BUILD_RAG = True` or when no pre-built index exists at `cfg.RAG_INDEX`.  
Safe to re-run; building takes ~30 s on a standard machine.

In [27]:
from src.rag import TabularRAG
from src.tools.tools_track_a import TrackAClient

rag = TabularRAG(k=3)
_rag_path = Path(cfg.RAG_INDEX)

if cfg.BUILD_RAG:
    print(f"Building RAG index from {cfg.TRAIN_FILE_A} …")
    _client_a_rag = TrackAClient(base_url=cfg.TOOL_SERVER_URL)
    rag.build_from_train(cfg.TRAIN_FILE_A, _client_a_rag)
    rag.save(str(_rag_path))
    print(f"RAG index saved to {_rag_path}")
elif _rag_path.exists():
    print(f"Loading RAG index from {_rag_path} …")
    rag.load(str(_rag_path))
    print("RAG index loaded.")
else:
    print(
        f"No RAG index at {_rag_path} and cfg.BUILD_RAG=False.\n"
        "Track A will run without few-shot retrieval examples.\n"
        "Set cfg.BUILD_RAG = True in Cell 3 and re-run cells 3→9 to build it."
    )

No RAG index at rag_index.pkl and cfg.BUILD_RAG=False.
Track A will run without few-shot retrieval examples.
Set cfg.BUILD_RAG = True in Cell 3 and re-run cells 3→9 to build it.


## Cell 10 — Run Benchmark

Initialises all components, runs the LangGraph orchestrator over `test_data`, and records per-scenario timing.

In [ ]:
import logging
import time

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG if cfg.VERBOSE else logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# ── Track A components ────────────────────────────────────────────────────────
from src.tools.tools_track_a import TrackAClient

client_a = TrackAClient(base_url=cfg.TOOL_SERVER_URL)

_tool_cache_path = Path(cfg.TOOL_CACHE)
if cfg.WARM_CACHE:
    _track_a_ids = [
        s["scenario_id"] for s in test_data
        if s.get("track") == "A" and s.get("scenario_id")
    ]
    print(f"Warming tool cache for {len(_track_a_ids)} Track A scenarios …")
    client_a.warm_all(_track_a_ids)
    client_a.save_cache(str(_tool_cache_path))
    print(f"Cache saved to {_tool_cache_path}")
elif _tool_cache_path.exists():
    print(f"Loading tool cache from {_tool_cache_path} …")
    client_a.load_cache(str(_tool_cache_path))

# ── Track B components ────────────────────────────────────────────────────────
from src.tools.parsers_track_b import TrackBClient

_daily_limit = 1000 if cfg.PHASE == "1" else 2000
client_b = TrackBClient(base_url=cfg.TOOL_SERVER_URL, daily_limit=_daily_limit)

# ── Compile LangGraph pipelines ───────────────────────────────────────────────
from src.agents.agents_track_a import build_graph_a
from src.agents.agents_track_b import build_graph_b

print("Compiling LangGraph pipelines …")
graph_a = build_graph_a(client=client_a, rag=rag)
graph_b = build_graph_b(client=client_b)
print("Pipelines compiled.")

# ── Orchestrate ───────────────────────────────────────────────────────────────
from src.orchestrator import Orchestrator

orc = Orchestrator(
    {
        "track_a_graph":  graph_a,
        "track_b_graph":  graph_b,
        "track_b_client": client_b,
        "daily_limit":    _daily_limit,
        "wandb_logger":   wl,          # ← streams results to W&B as they arrive
    }
)

print(f"\nStarting benchmark — {len(test_data)} scenario(s), ENV={cfg.ENV} …\n")
_t0 = time.perf_counter()
results = orc.run(test_data)
_elapsed = time.perf_counter() - _t0

print(f"\nBenchmark complete in {_elapsed:.1f}s  ({_elapsed / max(len(test_data), 1):.1f}s/scenario)")

# ── Flush W&B (logs final table + closes run) ─────────────────────────────────
wl.finish()


Compiling LangGraph pipelines …
Pipelines compiled.


11:07:45 [INFO] src.orchestrator: [Track A] Processing 80e3aa96-815d-4683-980c-16db42eab0ef
11:07:45 [INFO] src.orchestrator: [Track A] Processing f55a819f-3fb9-4c8f-8859-a5b1649ff2d5
11:07:45 [INFO] src.orchestrator: [Track A] Processing 2cd1f674-a411-4860-82c6-a16ba624b172
11:07:45 [INFO] src.orchestrator: [Track A] Processing 923c459f-2ead-4c18-a204-751574ed6ae3
11:07:45 [DEBUG] urllib3.connectionpool: Starting new HTTP connection (1): localhost:8000
11:07:45 [DEBUG] urllib3.connectionpool: Starting new HTTP connection (1): localhost:8000
11:07:45 [DEBUG] urllib3.connectionpool: Starting new HTTP connection (1): localhost:8000
11:07:45 [DEBUG] urllib3.connectionpool: Starting new HTTP connection (1): localhost:8000
11:07:45 [DEBUG] urllib3.connectionpool: Starting new HTTP connection (1): localhost:8000
11:07:45 [DEBUG] urllib3.connectionpool: Starting new HTTP connection (1): localhost:8000
11:07:45 [DEBUG] urllib3.connectionpool: Starting new HTTP connection (1): localhost:8000
11


Starting benchmark — 6 scenario(s), ENV=dev …



11:07:58 [DEBUG] httpcore.http11: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'application/x-ndjson'), (b'Date', b'Fri, 24 Apr 2026 11:07:58 GMT'), (b'Transfer-Encoding', b'chunked')])
11:07:58 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
11:07:58 [DEBUG] httpcore.http11: receive_response_body.started request=<Request [b'POST']>
11:08:37 [DEBUG] httpcore.http11: receive_response_body.complete
11:08:37 [DEBUG] httpcore.http11: response_closed.started
11:08:37 [DEBUG] httpcore.http11: response_closed.complete
11:08:37 [DEBUG] src.agents.agents_track_a: [Track A] [80e3aa96-815d-4683-980c-16db42eab0ef] LLM raw response:
Based on the provided features and domain rules, here is the step-by-step analysis:

1.  **Handover Status Analysis**:
    *   `handover_failure`: **True**
    *   `delta_db`: **10.86 dB**
    *   `threshold_db`: **4.0 dB**
    *   According to the **LATE_HANDOVER** rule: `handover_failur


Benchmark complete in 535.2s  (89.2s/scenario)


## Cell 11 — Results & Quick Metrics

In [29]:
import pandas as pd

# ── Build results dataframe ───────────────────────────────────────────────────
df_results = pd.DataFrame(results)
print(f"Results shape: {df_results.shape}")
df_results.head(20)

Results shape: (6, 3)


,ID,Track A,Track B
0,2cd1f674-a411-4860-82c6-a16ba624b172,C10,
1,2e83ae4f-dca0-4931-bb13-d13d672e14e1,C21,
2,80e3aa96-815d-4683-980c-16db42eab0ef,C1,
3,923c459f-2ead-4c18-a204-751574ed6ae3,C3,
4,cb846c3c-8900-4b40-917b-0bd848114d00,C4,
5,f55a819f-3fb9-4c8f-8859-a5b1649ff2d5,C11,


In [30]:
# ── Coverage metrics ──────────────────────────────────────────────────────────
# "Coverage" = fraction of rows that have a non-empty answer for each track.

def _coverage(series: pd.Series) -> float:
    valid = series.dropna().astype(str).str.strip()
    valid = valid[valid != ""]
    return len(valid) / max(len(series), 1)


print("=" * 50)
print("  Benchmark Summary")
print("=" * 50)
print(f"  Scenarios processed : {len(df_results)}")
print(f"  ENV                 : {cfg.ENV}")
print(f"  Model               : {settings.model_name}")
print(f"  Total time          : {_elapsed:.1f}s")
print(f"  Avg time/scenario   : {_elapsed / max(len(test_data), 1):.1f}s")
print()

for col in ["Track A", "Track B"]:
    if col in df_results.columns:
        cov = _coverage(df_results[col])
        non_empty = df_results[col].dropna().astype(str).str.strip().ne("").sum()
        print(f"  {col} — answered: {non_empty}/{len(df_results)}  ({cov:.0%} coverage)")

print("=" * 50)

  Benchmark Summary
  Scenarios processed : 6
  ENV                 : dev
  Model               : qwen3.5:35b-a3b
  Total time          : 535.2s
  Avg time/scenario   : 89.2s

  Track A — answered: 6/6  (100% coverage)
  Track B — answered: 0/6  (0% coverage)


In [31]:
# ── Compare against submission example (optional) ─────────────────────────────
_example_path = Path("data/raw/submission/Phase_1/submission_example.csv")

if _example_path.exists():
    df_example = pd.read_csv(_example_path)
    common_ids = set(df_results["ID"]) & set(df_example["ID"])
    print(f"Overlap with submission example: {len(common_ids)} / {len(df_results)} IDs\n")

    if common_ids:
        df_merged = df_results.set_index("ID").join(
            df_example.set_index("ID"),
            lsuffix="_pred",
            rsuffix="_ref",
        ).reset_index()

        for col in ["Track A", "Track B"]:
            pred_col = f"{col}_pred"
            ref_col  = f"{col}_ref"
            if pred_col in df_merged.columns and ref_col in df_merged.columns:
                match = (
                    df_merged[pred_col].fillna("").astype(str).str.strip() ==
                    df_merged[ref_col].fillna("").astype(str).str.strip()
                ).sum()
                total = len(df_merged)
                print(f"  {col} exact-match vs example : {match}/{total}  ({match/max(total,1):.0%})")
else:
    print("submission_example.csv not found — skipping comparison.")

Overlap with submission example: 6 / 6 IDs

  Track A exact-match vs example : 0/6  (0%)
  Track B exact-match vs example : 6/6  (100%)


## Cell 12 — Export Results

Saves `result.csv` and offers a one-click download on Colab / Kaggle.

In [32]:
from src.orchestrator import Orchestrator

output_path = cfg.OUTPUT_CSV
Orchestrator.write_csv(results, output_path)
print(f"Results written to {Path(output_path).resolve()}")

# ── Platform-specific download helpers ────────────────────────────────────────
if PLATFORM == "colab":
    from google.colab import files
    files.download(output_path)
    print("Download triggered in browser.")
elif PLATFORM == "kaggle":
    # On Kaggle, files in /kaggle/working are automatically available as output.
    import shutil
    _out = Path("/kaggle/working") / Path(output_path).name
    shutil.copy(output_path, _out)
    print(f"Copied to Kaggle output: {_out}")
else:
    print("Local run — file saved, no browser download needed.")

11:16:40 [INFO] src.orchestrator: result.csv written to result.csv (6 rows)


Results written to /content/Troubleshooting-Agentic/result.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered in browser.


## Cell 13 — (Optional) Cleanup

Stop any background servers started in Cell 7.

In [33]:
if "_server_proc" in dir() and _server_proc is not None:
    _server_proc.terminate()
    _server_proc.wait(timeout=5)
    print("Tool server stopped.")
else:
    print("No server process to stop.")

Tool server stopped.
